The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

accounts.py - made mostly by a Engineering Team using CrewAI

In [15]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [16]:
from accounts import Account

In [17]:
account = Account.get("Gaurav")
account

Account(name='gaurav', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [18]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "gaurav", "balance": 9897.796, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 34.068, "timestamp": "2026-05-28 11:16:32", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-28 11:16:32", 9987.796]], "total_portfolio_value": 9987.796, "total_profit_loss": -12.203999999999724}'

In [19]:
account.report()

'{"name": "gaurav", "balance": 9897.796, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 34.068, "timestamp": "2026-05-28 11:16:32", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-28 11:16:32", 9987.796], ["2026-05-28 11:16:39", 10182.796]], "total_portfolio_value": 10182.796, "total_profit_loss": 182.79600000000028}'

In [21]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 34.068,
  'timestamp': '2026-05-28 11:16:32',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [22]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [23]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [24]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Gaurav and my account is under the name Gaurav. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [25]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Gaurav, your current cash balance is $9,897.80. Your holdings include 3 shares of Amazon (AMZN). If you have any further questions or need assistance with your account, feel free to ask!

### Now let's build our own MCP Client

In [26]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [27]:
request = "My name is Gaurav and my account is under the name Gaurav. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your account balance is $9,897.80. Let me know if there is anything else you would like to check or do!

In [29]:
context = await read_accounts_resource("gaurav")
print(context)

{"name": "gaurav", "balance": 9897.796, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 34.068, "timestamp": "2026-05-28 11:16:32", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-28 11:16:32", 9987.796], ["2026-05-28 11:16:39", 10182.796], ["2026-05-28 11:19:08", 9990.796]], "total_portfolio_value": 9990.796, "total_profit_loss": -9.203999999999724}


In [30]:
from accounts import Account
Account.get("gaurav").report()

'{"name": "gaurav", "balance": 9897.796, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 34.068, "timestamp": "2026-05-28 11:16:32", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-05-28 11:16:32", 9987.796], ["2026-05-28 11:16:39", 10182.796], ["2026-05-28 11:19:08", 9990.796], ["2026-05-28 11:19:15", 10056.796]], "total_portfolio_value": 10056.796, "total_profit_loss": 56.79600000000028}'